# Solution 6 — complete UR5e calibration, MHE, and control

This worked solution completes every task: local DH geometry, both Jacobian
checks, training/validation calibration, state estimation, obstacle-avoiding
OCPs on two meshes with sparsity and dense-clearance comparisons, full state/input
plots, and 24 moves of estimated-state MPC. It runs independently of all chapter
notebooks and never imports a model formulation from a package.

### Prepare the notebook and optional interactive frontend

Imports below provide array operations, CasADi graphs, and presentation only.
Every robot model and optimization formulation is defined in this notebook.
Plotly gives hover/zoom plots when installed; Matplotlib is the fallback.
Widgets use explicit **Run** and **Reset** buttons, so moving a slider cannot
launch a solve. `CASADI_TUTORIAL_WIDGETS=0` disables widgets for batch execution.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

repository_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_robot, plot_series
from casadi_tutorial.diagnostics import bound_report, max_bound_violation, solver_summary

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

np.set_printoptions(precision=5, suppress=True)

### Write the robot model: frames, parameters, and coordinates

The table below is the [UR5e standard DH geometry published by Universal Robots](https://www.universal-robots.com/articles/ur/application-installation/dh-parameters-for-calculations-of-kinematics-and-dynamics/).
Rows follow joints 1–6 from base to tool; all lengths are metres and all angles are radians.

| Joint | $a_i$ | $d_i$ | $\alpha_i$ |
|---|---:|---:|---:|
| 1 | 0 | 0.1625 | $\pi/2$ |
| 2 | −0.425 | 0 | 0 |
| 3 | −0.3922 | 0 | 0 |
| 4 | 0 | 0.1333 | $\pi/2$ |
| 5 | 0 | 0.0997 | $-\pi/2$ |
| 6 | 0 | 0.0996 | 0 |

For **standard** DH, the relative transform is

$$
{}^{i-1}T_i=\begin{bmatrix}
c_i&-s_i c_{\alpha_i}&s_i s_{\alpha_i}&a_i c_i\\
s_i&c_i c_{\alpha_i}&-c_i s_{\alpha_i}&a_i s_i\\
0&s_{\alpha_i}&c_{\alpha_i}&d_i\\0&0&0&1
\end{bmatrix},\qquad
T^0_6=\prod_{i=1}^6{}^{i-1}T_i=\begin{bmatrix}R&p\\0&1\end{bmatrix}.
$$

Calibration will adjust only $\theta=[\Delta a_2,\Delta a_3,\delta_2,\delta_3]$.
Here $q$ always means the six **encoder coordinates**, and
$q_{\rm physical}=q+[0,\delta_2,\delta_3,0,0,0]^T$.
The offset is added exactly once, inside this model. Fixed base and tool frames are assumed known.
`pose_map(q, theta)` returns $4\times4$; `position_map` returns $3\times1$;
`origins_map` returns $3\times7$. Joint, speed, and acceleration limits below are conservative teaching choices.
These functions are defined locally so the whole graph can be inspected and edited here.

In [ ]:
# Standard DH, metres and radians. The table is included here rather than imported.
dh_a = np.array([0.0, -0.425, -0.3922, 0.0, 0.0, 0.0])
dh_d = np.array([0.1625, 0.0, 0.0, 0.1333, 0.0997, 0.0996])
dh_alpha = np.array([np.pi / 2, 0.0, 0.0, np.pi / 2, -np.pi / 2, 0.0])
q_min_physical = np.full(6, -2 * np.pi)
q_max_physical = np.full(6, 2 * np.pi)
v_max = np.full(6, np.pi)
u_max = np.full(6, 2 * np.pi)


def encoder_bias(theta):
    """theta = [delta_a2, delta_a3, delta_encoder2, delta_encoder3]."""
    return ca.vertcat(0, theta[2], theta[3], 0, 0, 0)


def dh_transform(angle, distance, length, twist):
    """One STANDARD-DH transform RotZ(angle) TransZ(d) TransX(a) RotX(alpha)."""
    c, s = ca.cos(angle), ca.sin(angle)
    ct, st = np.cos(twist), np.sin(twist)
    return ca.vertcat(
        ca.horzcat(c, -s * ct, s * st, length * c),
        ca.horzcat(s, c * ct, -c * st, length * s),
        ca.horzcat(0, st, ct, distance),
        ca.horzcat(0, 0, 0, 1),
    )


def link_transforms(q_encoder, theta=None):
    """Seven base-to-frame transforms, including the identity base frame."""
    theta = ca.DM.zeros(4) if theta is None else theta
    physical_q = q_encoder + encoder_bias(theta)
    lengths = ca.vertcat(dh_a[0], dh_a[1] + theta[0], dh_a[2] + theta[1], *dh_a[3:])
    transforms = [ca.DM.eye(4)]
    for joint in range(6):
        transforms.append(
            transforms[-1] @ dh_transform(
                physical_q[joint], dh_d[joint], lengths[joint], dh_alpha[joint]
            )
        )
    return transforms


def forward_kinematics(q_encoder, theta=None):
    return link_transforms(q_encoder, theta)[-1]


def frame_origins(q_encoder, theta=None):
    return ca.horzcat(*[transform[:3, 3] for transform in link_transforms(q_encoder, theta)])


q_symbol = ca.SX.sym("q_encoder", 6)
theta_symbol = ca.SX.sym("theta", 4)
pose_map = ca.Function("pose", [q_symbol, theta_symbol], [forward_kinematics(q_symbol, theta_symbol)])
position_map = ca.Function("position", [q_symbol, theta_symbol], [pose_map(q_symbol, theta_symbol)[:3, 3]])
origins_map = ca.Function("origins", [q_symbol, theta_symbol], [frame_origins(q_symbol, theta_symbol)])
nominal_theta = np.zeros(4)

### Derive the geometric Jacobian instead of importing it

All joint axes and origins below are expressed in the **base frame**. For revolute joint $i$,
the column of the $6\times6$ geometric Jacobian is

$$J_i(q)=\begin{bmatrix}z_{i-1}\times(p_6-p_{i-1})\\z_{i-1}\end{bmatrix},
\qquad \begin{bmatrix}v_{\rm tool}\\\omega_{\rm tool}\end{bmatrix}=J_g(q)\dot q.$$

The translational block should equal `ca.jacobian(position, q)`. Angular velocity
is expressed in the base frame too; it is not an Euler-angle derivative.
Keeping both constructions gives an independent check of frame indexing.

In [ ]:
def geometric_jacobian(q_encoder, theta=None):
    transforms = link_transforms(q_encoder, theta)
    tool = transforms[-1][:3, 3]
    columns = []
    for joint in range(6):
        axis = transforms[joint][:3, 2]
        origin = transforms[joint][:3, 3]
        columns.append(ca.vertcat(ca.cross(axis, tool - origin), axis))
    return ca.horzcat(*columns)


jacobian_map = ca.Function(
    "geometric_jacobian", [q_symbol, theta_symbol],
    [geometric_jacobian(q_symbol, theta_symbol)],
)
ad_position_jacobian = ca.Function(
    "position_jacobian_ad", [q_symbol, theta_symbol],
    [ca.jacobian(position_map(q_symbol, theta_symbol), q_symbol)],
)

### Check the model before optimizing

This cell checks a known pose, homogeneous coordinates, rotation orthogonality, and
both Jacobian blocks at six configurations. With $\varepsilon=10^{-6}$, compare

$$\frac{p(q+\varepsilon d)-p(q-\varepsilon d)}{2\varepsilon}\approx J_v(q)d.$$

For orientation, $[\omega]_\times=\dot R R^T$; extracting the skew entries checks
the angular block without subtracting Euler angles. Errors should be near numerical
roundoff. The zero configuration is singular, so rank below six is expected there.
A successful shape assertion alone would not catch an incorrect frame convention.

In [ ]:
zero_pose = np.asarray(pose_map(np.zeros(6), nominal_theta))
np.testing.assert_allclose(zero_pose[:3, 3], [-0.8172, -0.2329, 0.0628], atol=1e-10)
assert zero_pose.shape == (4, 4)
rng_checks = np.random.default_rng(16)
max_direction_error = 0.0
for configuration in [np.zeros(6), *rng_checks.uniform(-1, 1, (5, 6))]:
    transform = np.asarray(pose_map(configuration, nominal_theta))
    rotation = transform[:3, :3]
    np.testing.assert_allclose(rotation.T @ rotation, np.eye(3), atol=1e-12)
    np.testing.assert_allclose(np.linalg.det(rotation), 1.0, atol=1e-12)
    np.testing.assert_allclose(transform[3], [0, 0, 0, 1], atol=1e-12)
    jacobian = np.asarray(jacobian_map(configuration, nominal_theta))
    np.testing.assert_allclose(jacobian[:3], ad_position_jacobian(configuration, nominal_theta), atol=1e-12)
    direction = rng_checks.normal(size=6)
    epsilon = 1e-6
    plus = np.asarray(pose_map(configuration + epsilon * direction, nominal_theta))
    minus = np.asarray(pose_map(configuration - epsilon * direction, nominal_theta))
    finite_linear = (plus[:3, 3] - minus[:3, 3]) / (2 * epsilon)
    rotation_rate = (plus[:3, :3] - minus[:3, :3]) / (2 * epsilon)
    skew_omega = rotation_rate @ rotation.T
    finite_angular = np.array([skew_omega[2, 1], skew_omega[0, 2], skew_omega[1, 0]])
    error = np.max(np.abs(np.r_[finite_linear, finite_angular] - jacobian @ direction))
    max_direction_error = max(max_direction_error, error)
    np.testing.assert_allclose(np.r_[finite_linear, finite_angular], jacobian @ direction, atol=2e-8, rtol=2e-6)
print("Known zero-pose tool position [m]:", zero_pose[:3, 3])
print("Maximum linear/angular directional-check error:", max_direction_error)
print("Zero-pose Jacobian rank:", np.linalg.matrix_rank(np.asarray(jacobian_map(np.zeros(6), nominal_theta))))

### Create independent pose measurements and reserve validation data

A synthetic tracker observes the complete tool pose at 36 diverse, known encoder
configurations. The simulated plant has 4 mm and −3 mm link corrections and small
joint-2/3 zero offsets. **Only data generation sees `theta_truth`.** The estimator
starts at nominal parameters. Tracker noise is 0.5 mm per Cartesian component
and 0.002 rad per rotation-vector component.

Twenty-four randomly selected poses train the estimator; the remaining twelve
are held out. The split and random seed stay fixed when changing noise, allowing
fair comparisons. Tool/base extrinsics, other DH constants, and encoder noise
during this static calibration experiment are assumed known. Estimating every
DH entry and frame offset together would introduce ambiguity.

In [ ]:
from scipy.spatial.transform import Rotation

parameter_scale = np.array([0.01, 0.01, 0.05, 0.05])
theta_truth = np.array([0.004, -0.003, 0.015, -0.012])
sigma_position = 0.0005
sigma_rotation = 0.002


def calibration_dataset(noise_multiplier=1.0):
    rng = np.random.default_rng(2026)
    configurations = rng.uniform(
        [-1.5, -1.5, 0.2, -1.0, 0.4, -1.5],
        [1.5, -0.2, 1.7, 1.0, 1.6, 1.5], (36, 6),
    )
    measured = np.array([np.asarray(pose_map(q, theta_truth)) for q in configurations])
    measured[:, :3, 3] += rng.normal(0, sigma_position * noise_multiplier, (36, 3))
    for transform in measured:
        rotation_noise = Rotation.from_rotvec(rng.normal(0, sigma_rotation * noise_multiplier, 3)).as_matrix()
        transform[:3, :3] = transform[:3, :3] @ rotation_noise
    indices = rng.permutation(36)
    return {
        "q": configurations, "poses": measured,
        "training": indices[:24], "validation": indices[24:],
        "noise_multiplier": noise_multiplier,
    }


calibration_data = calibration_dataset()
print("Training / validation poses:", len(calibration_data["training"]), "/", len(calibration_data["validation"]))

### Formulate and solve a scaled nonlinear least-squares calibration

With $\theta=D z$, $D=\operatorname{diag}(0.01\,\mathrm m,0.01\,\mathrm m,
0.05\,\mathrm{rad},0.05\,\mathrm{rad})$, solve

$$\min_{-1\le z\le1}\sum_{j\in\mathcal T}\left\|
\begin{bmatrix}
(p(q_j;Dz)-p_j)/\sigma_p\\
\operatorname{vec}(R_j^TR(q_j;Dz)-I)/(\sqrt2\sigma_R)
\end{bmatrix}\right\|_2^2.$$

The rotation term is a smooth **chordal rotation residual**:
$\|R_j^TR-I\|_F^2/2=4\sin^2(\phi/2)\approx\phi^2$ for small relative angle
$\phi$. Its nine entries encode three rotation degrees of freedom; this is a
weighted least-squares teaching objective, not an exact independent-component likelihood.
Small, bounded corrections keep the problem close to the measured frame convention.
The $12|\mathcal T|\times4$ residual Jacobian is differentiated with respect to
scaled parameters so length and angle columns can be compared sensibly.

In [ ]:
def calibration_residual(q_observed, pose_observed, theta, noise_multiplier=1.0):
    prediction = pose_map(q_observed, theta)
    position_residual = (prediction[:3, 3] - pose_observed[:3, 3]) / (sigma_position * noise_multiplier)
    relative_rotation = pose_observed[:3, :3].T @ prediction[:3, :3]
    rotation_residual = ca.vec(relative_rotation - ca.DM.eye(3)) / (np.sqrt(2) * sigma_rotation * noise_multiplier)
    return ca.vertcat(position_residual, rotation_residual)


def fit_calibration(data, training_count=24):
    problem = ca.Opti()
    normalized_theta = problem.variable(4)
    theta = ca.DM(parameter_scale) * normalized_theta
    residual = ca.vertcat(*[
        calibration_residual(data["q"][index], ca.DM(data["poses"][index]), theta, data["noise_multiplier"])
        for index in data["training"][:training_count]
    ])
    problem.minimize(ca.sumsqr(residual))
    problem.subject_to(problem.bounded(-1, normalized_theta, 1))
    problem.set_initial(normalized_theta, 0)
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 100, "tol": 1e-9})
    solution = problem.solve()
    estimated_theta = np.asarray(solution.value(theta)).ravel()
    sensitivity = ca.Function("calibration_sensitivity", [normalized_theta], [ca.jacobian(residual, normalized_theta)])
    singular_values = np.linalg.svd(np.asarray(sensitivity(estimated_theta / parameter_scale)), compute_uv=False)
    return {"theta": estimated_theta, "singular_values": singular_values, "stats": solution.stats(), "training_count": training_count}


calibration_result = fit_calibration(calibration_data)
theta_estimate = calibration_result["theta"]
print("[delta a2 [m], delta a3 [m], delta2 [rad], delta3 [rad]]")
print("estimated:", theta_estimate)
print("truth (evaluation only):", theta_truth)

### Validate prediction and identifiability on data the solver did not see

Report position and rotation-angle RMSE independently on training and held-out
poses. Good training loss alone can conceal a poor model. The scaled sensitivity
must have four non-negligible singular values; otherwise some parameter combination
cannot be resolved by these poses. The default held-out position error should
fall from roughly 10 mm to roughly 1 mm, near the synthetic noise level.

Parameter-recovery assertions below use simulation truth **only after fitting**.
Real calibration must instead rely on held-out measurements, uncertainty analysis,
and independent geometric checks. Bounds alone do not make a parameter identifiable.

In [ ]:
def pose_errors(data, theta, indices):
    position_errors, rotation_errors = [], []
    for index in indices:
        prediction = np.asarray(pose_map(data["q"][index], theta))
        measurement = data["poses"][index]
        position_errors.append(np.linalg.norm(prediction[:3, 3] - measurement[:3, 3]))
        rotation_errors.append(Rotation.from_matrix(measurement[:3, :3].T @ prediction[:3, :3]).magnitude())
    return np.asarray(position_errors), np.asarray(rotation_errors)


def summarize_calibration(data, result):
    metrics = {}
    for split in ("training", "validation"):
        indices = data[split][:result["training_count"]] if split == "training" else data[split]
        for label, theta in (("nominal", nominal_theta), ("calibrated", result["theta"])):
            position_error, rotation_error = pose_errors(data, theta, indices)
            metrics[split, label] = (np.sqrt(np.mean(position_error**2)), np.sqrt(np.mean(rotation_error**2)))
            print(f"{split:10s} {label:10s}: position RMSE {metrics[split, label][0] * 1000:.3f} mm; orientation RMSE {metrics[split, label][1]:.5f} rad")
    singular_values = result["singular_values"]
    print("Scaled sensitivity singular values:", singular_values)
    print("Scaled sensitivity condition number:", singular_values[0] / singular_values[-1])
    return metrics


calibration_metrics = summarize_calibration(calibration_data, calibration_result)
assert calibration_result["stats"]["success"]
assert calibration_result["singular_values"][-1] > 1e-6 * calibration_result["singular_values"][0]
assert calibration_metrics["validation", "calibrated"][0] < 0.002
assert calibration_metrics["validation", "calibrated"][0] < calibration_metrics["validation", "nominal"][0]
np.testing.assert_allclose(theta_estimate[:2], theta_truth[:2], atol=0.001)
np.testing.assert_allclose(theta_estimate[2:], theta_truth[2:], atol=0.002)

### Derive the acceleration-controlled dynamics and discrete step

The state is $x=[q^T,v^T]^T\in\mathbb R^{12}$ in encoder coordinates;
the input is commanded joint acceleration $u\in\mathbb R^6$. Constant zero offsets
do not change $\dot q=v$. For piecewise constant input over $h=0.2$ s,

$$\dot q=v,\quad\dot v=u,\qquad
x_{k+1}=A_dx_k+B_du_k,\quad
A_d=\begin{bmatrix}I&hI\\0&I\end{bmatrix},\quad
B_d=\begin{bmatrix}\tfrac12h^2I\\hI\end{bmatrix}.$$

This discretization is exact for this surrogate. The model contains no masses,
gravity, torques, drive dynamics, or contact. Physical angle limits must be shifted
by the estimated encoder bias before constraining encoder states.

In [ ]:
sample_time = 0.2


def discrete_matrices(step):
    identity = np.eye(6)
    a_matrix = np.block([[identity, step * identity], [np.zeros((6, 6)), identity]])
    b_matrix = np.vstack([0.5 * step**2 * identity, step * identity])
    return a_matrix, b_matrix


def exact_step(state, acceleration, step=sample_time):
    return ca.vertcat(
        state[:6] + step * state[6:] + 0.5 * step**2 * acceleration,
        state[6:] + step * acceleration,
    )


a_discrete, b_discrete = discrete_matrices(sample_time)
x_symbol = ca.SX.sym("state", 12)
u_symbol = ca.SX.sym("acceleration", 6)
step_map = ca.Function("exact_double_integrator", [x_symbol, u_symbol], [exact_step(x_symbol, u_symbol)])
np.testing.assert_allclose(step_map(np.zeros(12), np.ones(6)), np.r_[np.full(6, 0.5 * sample_time**2), np.full(6, sample_time)][:, None])
q_min_encoder = q_min_physical - np.asarray(encoder_bias(theta_estimate)).ravel()
q_max_encoder = q_max_physical - np.asarray(encoder_bias(theta_estimate)).ravel()

### Reject an invalid solve before using an estimate or applying a command

Solver termination is one check. A candidate must also contain finite decisions
and satisfy every original Opti constraint within $10^{-5}$ in that row's units.
The helpers below measure wall time, retain status and iteration count, and check
$\max(0,l_g-g,g-u_g)$. On failure, print the worst rows and `g_describe` source
locations, then stop. Later cells independently check dynamics and input bounds
before a predicted acceleration reaches the simulated plant.

Residual units differ between constraints; this compact maximum is a rejection
gate, not a physically normalized accuracy score. Small equality residuals or
successful status alone cannot establish a useful or collision-free trajectory.

In [ ]:
def constraint_diagnostics(problem, candidate):
    values = candidate.value(problem.g)
    lower = candidate.value(problem.lbg)
    upper = candidate.value(problem.ubg)
    violation = max_bound_violation(values, lower, upper)
    return violation, bound_report(values, lower, upper, labels=getattr(problem, "tutorial_constraint_labels", None))


def checked_solve(problem, label, tolerance=1e-5):
    started = perf_counter()
    try:
        solution = problem.solve()
    except RuntimeError:
        print(label, "failed:", solver_summary(problem.stats(), elapsed_s=perf_counter() - started))
        violation, rows = constraint_diagnostics(problem, problem.debug)
        for row in sorted(rows, key=lambda item: item["violation"], reverse=True)[:3]:
            print("Constraint diagnostic:", row)
            print(problem.debug.g_describe(row["index"]))
        raise
    report = solver_summary(solution.stats(), elapsed_s=perf_counter() - started)
    report["constraint_violation"], _ = constraint_diagnostics(problem, solution)
    report["finite"] = bool(np.isfinite(np.asarray(solution.value(problem.x))).all())
    report["accepted"] = bool(report["success"] and report["finite"] and report["constraint_violation"] <= tolerance)
    if not report["accepted"]:
        print(label, "candidate rejected:", report)
        raise RuntimeError(label + " did not pass the numerical acceptance checks")
    return solution, report

### Formulate moving-horizon state estimation from encoders and a tool tracker

Over $L=6$ intervals, estimate seven 12-state vectors and six 6-axis acceleration
disturbances: **120 decision variables**. The known commands are $u_i$; noisy
observations are $y^q_i=q_i+\epsilon^q_i$ and
$y^p_i=p(q_i;\hat\theta)+\epsilon^p_i$. Solve

$$\min_{X,W}\|x_0-\bar x_0\|_{P^{-1}}^2+
\sum_{i=0}^{L}\left(\left\|\frac{q_i-y_i^q}{\sigma_q}\right\|^2+
\left\|\frac{p(q_i;\hat\theta)-y_i^p}{\sigma_p}\right\|^2\right)
+\sum_{i=0}^{L-1}\left\|\frac{w_i}{\sigma_a}\right\|^2,$$
$$\text{subject to }x_{i+1}=A_dx_i+B_d(u_i+w_i).$$

Use $\sigma_q=0.003$ rad, $\sigma_p=0.001$ m, and $\sigma_a=0.08$ rad/s².
The arrival standard deviations are 0.01 rad and 0.08 rad/s. The prior summarizes
earlier history approximately; a fixed diagonal arrival cost is not a full
covariance marginalization. Position observations over time and the dynamics
identify velocity; a single encoder observation does not.

Each new solve shifts the previous state trajectory and predicts one extra node.
The calibrated FK enters the tool residual, while encoder offsets are never added
to encoder observations. Write `states[:6, :]` explicitly: a single CasADi slice
indexes a flattened matrix rather than selecting all six rows across time.

In [ ]:
def build_mhe(theta, window=6, encoder_sigma=0.003, tool_sigma=0.001, acceleration_sigma=0.08):
    problem = ca.Opti()
    states = problem.variable(12, window + 1)
    disturbances = problem.variable(6, window)
    inputs = problem.parameter(6, window)
    encoders = problem.parameter(6, window + 1)
    tools = problem.parameter(3, window + 1)
    arrival = problem.parameter(12)
    arrival_sigma = ca.DM([0.01] * 6 + [0.08] * 6)
    objective = ca.sumsqr((states[:, 0] - arrival) / arrival_sigma)
    objective += ca.sumsqr((states[:6, :] - encoders) / encoder_sigma)
    objective += ca.sumsqr(disturbances / acceleration_sigma)
    for index in range(window + 1):
        objective += ca.sumsqr((position_map(states[:6, index], theta) - tools[:, index]) / tool_sigma)
    for index in range(window):
        problem.subject_to(states[:, index + 1] == exact_step(states[:, index], inputs[:, index] + disturbances[:, index]))
    problem.tutorial_constraint_labels = [f"MHE dynamics interval {index}, state {component}" for index in range(window) for component in range(12)]
    problem.minimize(objective)
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 100, "tol": 1e-7})
    return {"opti": problem, "X": states, "W": disturbances, "U": inputs, "Yq": encoders, "Yp": tools, "arrival": arrival, "window": window}


def estimate_window(estimator, inputs, encoders, tools, arrival, previous_states=None):
    problem = estimator["opti"]
    problem.set_value(estimator["U"], inputs)
    problem.set_value(estimator["Yq"], encoders)
    problem.set_value(estimator["Yp"], tools)
    problem.set_value(estimator["arrival"], arrival)
    if previous_states is None:
        initial_states = np.vstack([encoders, np.zeros_like(encoders)])
    else:
        predicted_end = a_discrete @ previous_states[:, -1] + b_discrete @ inputs[:, -1]
        initial_states = np.column_stack([previous_states[:, 1:], predicted_end])
    problem.set_initial(estimator["X"], initial_states)
    problem.set_initial(estimator["W"], 0)
    solution, report = checked_solve(problem, "MHE")
    states = solution.value(estimator["X"])
    disturbances = solution.value(estimator["W"])
    defects = states[:, 1:] - a_discrete @ states[:, :-1] - b_discrete @ (inputs + disturbances)
    report["dynamics_defect"] = float(np.max(np.abs(defects)))
    if report["dynamics_defect"] > 1e-5:
        raise RuntimeError("MHE dynamics check failed before feeding MPC")
    return states[:, -1], states, report

### Test the estimator against a separate disturbed plant

The plant uses true geometry and random acceleration disturbances; the estimator
uses **fitted** calibration parameters and receives only commanded accelerations,
encoder samples, and tracker samples. Truth is retained solely for scoring.
The first six intervals fill the observation window; reported estimates start
after that warm-up rather than inventing a full history at the first sample.

The baseline velocity is the backward encoder difference
$(y^q_k-y^q_{k-1})/h$. Differencing amplifies measurement noise and also represents
an interval-average velocity, while MHE estimates the current endpoint velocity.
The default should improve velocity RMSE substantially. A longer window or a
stronger arrival prior is not automatically better when the process model is wrong.

In [ ]:
def state_estimation_data(noise_multiplier=1.0, intervals=24):
    rng = np.random.default_rng(42)
    commands = np.array([0.35 * np.sin(np.arange(intervals) * 0.25 + joint * 0.6) for joint in range(6)])
    states = np.zeros((12, intervals + 1))
    states[:6, 0] = [0.1, -0.5, 0.6, -0.2, 0.7, 0.1]
    states[6:, 0] = [0.1, -0.1, 0, 0, 0, 0]
    for index in range(intervals):
        disturbance = rng.normal(0, 0.08, 6)
        states[:, index + 1] = a_discrete @ states[:, index] + b_discrete @ (commands[:, index] + disturbance)
    encoders = states[:6] + rng.normal(0, 0.003 * noise_multiplier, (6, intervals + 1))
    tools = np.array([np.asarray(position_map(q, theta_truth)).ravel() for q in states[:6].T]).T
    tools += rng.normal(0, 0.001 * noise_multiplier, tools.shape)
    return {"truth": states, "u": commands, "q_measured": encoders, "p_measured": tools}


def run_state_estimation(data, window=6, noise_multiplier=1.0):
    estimator = build_mhe(theta_estimate, window, 0.003 * noise_multiplier, 0.001 * noise_multiplier)
    arrival = np.r_[data["q_measured"][:, 0], np.zeros(6)]
    previous_states = None
    estimates, statuses = [], []
    for index in range(window, data["truth"].shape[1]):
        start = index - window
        estimate, fitted_states, stats = estimate_window(
            estimator, data["u"][:, start:index],
            data["q_measured"][:, start:index + 1], data["p_measured"][:, start:index + 1],
            arrival, previous_states,
        )
        estimates.append(estimate)
        statuses.append(stats["success"])
        arrival, previous_states = fitted_states[:, 1], fitted_states
    estimates = np.asarray(estimates).T
    truth = data["truth"][:, window:]
    raw_velocity = np.diff(data["q_measured"], axis=1)[:, window - 1:] / sample_time
    return {
        "estimates": estimates, "truth": truth, "raw_velocity": raw_velocity,
        "time": sample_time * np.arange(window, data["truth"].shape[1]),
        "position_rmse": np.sqrt(np.mean((estimates[:6] - truth[:6])**2)),
        "velocity_rmse": np.sqrt(np.mean((estimates[6:] - truth[6:])**2)),
        "raw_velocity_rmse": np.sqrt(np.mean((raw_velocity - truth[6:])**2)),
        "success": all(statuses), "window": window,
    }


state_data = state_estimation_data()
state_result = run_state_estimation(state_data)
print("Encoder-coordinate position RMSE [rad]:", state_result["position_rmse"])
print("Velocity RMSE: finite difference / MHE [rad/s]:", state_result["raw_velocity_rmse"], "/", state_result["velocity_rmse"])
assert state_result["success"] and np.isfinite(state_result["estimates"]).all()
assert state_result["velocity_rmse"] < state_result["raw_velocity_rmse"]

### Compare measured, estimated, and true states without rerunning the solver

Start with joint 2, where the robot moves visibly. The position plot overlays
encoder observations with truth and MHE; the velocity plot contrasts noisy
differencing with the dynamic estimate. Slider changes select a joint from
cached results. To change noise or window length, use the separate experiment
button below. A delayed or oversmoothed estimate is visible even when aggregate
RMSE looks acceptable.

In [ ]:
def show_estimation_joint(joint=2):
    index = int(joint) - 1
    start = state_result["window"]
    display_plot(plot_series(state_result["time"], {
        "truth": state_result["truth"][index],
        "encoder": state_data["q_measured"][index, start:],
        "MHE": state_result["estimates"][index],
    }, title=f"Joint {joint}: position estimation", ylabel="q_encoder [rad]"))
    display_plot(plot_series(state_result["time"], {
        "truth": state_result["truth"][6 + index],
        "backward difference": state_result["raw_velocity"][index],
        "MHE": state_result["estimates"][6 + index],
    }, title=f"Joint {joint}: velocity estimation", ylabel="v [rad/s]"))


show_estimation_joint()
joint_control = widgets.IntSlider(value=2, min=1, max=6, description="Joint") if widgets else 2
experiment(show_estimation_joint, {"joint": joint_control}, title="Inspect another cached joint")

### Formulate a rest-to-rest reference OCP with an active obstacle

The target posture is specified in physical coordinates and converted to encoder
coordinates using $\hat\theta$. Place a 7 cm sphere **on** the straight joint-path
tool trace; this makes avoidance a real requirement. A 1.5 cm planning margin
allows for discretization, calibration, and tracking errors in this demonstration.
The obstacle still concerns only the tool point, not the robot's links.

For $N$ intervals over 3 s, optimize $X\in\mathbb R^{12\times(N+1)}$,
$U\in\mathbb R^{6\times N}$ and dimensionless $S\in\mathbb R^{1\times(N+1)}$:

$$\min h\sum_k\|u_k/2\|^2+0.01h\sum_k w_k\|v_k\|^2
+10^5h\sum_k w_ks_k^2$$
$$x_{k+1}=A_dx_k+B_du_k,\quad
x_0=[q_0;0],\quad x_N=[q_g;0],\quad
q_{\min}\le q_k\le q_{\max},\quad |v_k|\le v_{\max},\quad |u_k|\le u_{\max},$$
$$\frac{\|p(q_k;\hat\theta)-c\|^2}{r_{\rm plan}^2}+s_k\ge1,\qquad s_k\ge0.$$

Also check each interval midpoint with its interval's slack. Time weights
$w_0=w_N=1/2$ make state/slack costs comparable across meshes. A slack is an
allowed relaxation, **not evidence of clearance**. The initial guess is an arcing
trajectory whose accelerations are projected onto exact endpoint conditions;
starting at the sphere center would give a zero collision-distance gradient.
The functions below expose every constraint and cost locally for reuse in mesh experiments.

In [ ]:
q_initial = np.zeros(6)
q_goal_physical = np.array([0, -np.pi / 3, np.pi / 2, -np.pi / 3, np.pi / 2, 0])
q_goal = q_goal_physical - np.asarray(encoder_bias(theta_estimate)).ravel()
duration = 3.0
obstacle_radius = 0.07
planning_radius = obstacle_radius + 0.015
# Placing the sphere ON the nominal joint-interpolation path makes it active.
obstacle_center = np.asarray(position_map(0.5 * (q_initial + q_goal), theta_estimate)).ravel()


def feasible_seed(nodes, total_time=duration, bump=0.4):
    h = total_time / nodes
    fractions = (np.arange(nodes) + 0.5) / nodes
    shape_acceleration = (60 * fractions - 180 * fractions**2 + 120 * fractions**3) / total_time**2
    accelerations = (q_goal - q_initial)[:, None] * shape_acceleration
    accelerations[0] += bump * 2 * np.pi**2 * np.cos(2 * np.pi * fractions) / total_time**2
    endpoint_matrix = np.vstack([h**2 * (nodes - np.arange(nodes) - 0.5), np.full(nodes, h)])
    endpoint_target = np.column_stack([q_goal - q_initial, np.zeros(6)])
    accelerations += (endpoint_target - accelerations @ endpoint_matrix.T) @ np.linalg.solve(endpoint_matrix @ endpoint_matrix.T, endpoint_matrix)
    states = np.zeros((12, nodes + 1))
    states[:6, 0] = q_initial
    a_matrix, b_matrix = discrete_matrices(h)
    for index in range(nodes):
        states[:, index + 1] = a_matrix @ states[:, index] + b_matrix @ accelerations[:, index]
    return states, accelerations


def build_reference_ocp(nodes=15, total_time=duration):
    h = total_time / nodes
    problem = ca.Opti()
    states = problem.variable(12, nodes + 1)
    inputs = problem.variable(6, nodes)
    slack = problem.variable(1, nodes + 1)
    problem.subject_to(states[:, 0] == ca.vertcat(q_initial, ca.DM.zeros(6)))
    problem.subject_to(states[:, -1] == ca.vertcat(q_goal, ca.DM.zeros(6)))
    problem.subject_to(problem.bounded(q_min_encoder[:, None], states[:6, :], q_max_encoder[:, None]))
    problem.subject_to(problem.bounded(-v_max[:, None], states[6:, :], v_max[:, None]))
    problem.subject_to(problem.bounded(-u_max[:, None], inputs, u_max[:, None]))
    problem.subject_to(slack >= 0)
    for index in range(nodes):
        problem.subject_to(states[:, index + 1] == exact_step(states[:, index], inputs[:, index], h))
        midpoint = exact_step(states[:, index], inputs[:, index], 0.5 * h)
        normalized_distance = ca.sumsqr(position_map(midpoint[:6], theta_estimate) - obstacle_center) / planning_radius**2
        problem.subject_to(normalized_distance + slack[0, index] >= 1)
    for index in range(nodes + 1):
        normalized_distance = ca.sumsqr(position_map(states[:6, index], theta_estimate) - obstacle_center) / planning_radius**2
        problem.subject_to(normalized_distance + slack[0, index] >= 1)
    # Trapezoidal state/slack quadrature; interval-held input cost uses rectangle quadrature.
    node_weights = ca.DM([0.5, *([1.0] * (nodes - 1)), 0.5]).T
    objective = h * ca.sumsqr(inputs / 2.0)
    objective += 0.01 * h * ca.sum2(ca.sum1(states[6:, :]**2) * node_weights)
    objective += 1e5 * h * ca.sum2(slack**2 * node_weights)
    problem.minimize(objective)
    initial_states, initial_inputs = feasible_seed(nodes, total_time)
    problem.set_initial(states, initial_states)
    problem.set_initial(inputs, initial_inputs)
    problem.set_initial(slack, 0)
    labels = ["initial state"] * 12 + ["terminal state"] * 12
    labels += ["joint angle bound"] * (6 * (nodes + 1)) + ["joint velocity bound"] * (6 * (nodes + 1))
    labels += ["acceleration bound"] * (6 * nodes) + ["nonnegative obstacle slack"] * (nodes + 1)
    for index in range(nodes):
        labels += [f"dynamics interval {index}"] * 12 + [f"midpoint obstacle interval {index}"]
    labels += [f"tool obstacle node {index}" for index in range(nodes + 1)]
    problem.tutorial_constraint_labels = labels
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 500, "tol": 1e-8})
    return {"opti": problem, "X": states, "U": inputs, "S": slack, "h": h, "nodes": nodes}


def solve_reference_ocp(nodes=15, warm_start=None):
    graph = build_reference_ocp(nodes)
    if warm_start is not None:
        new_times = np.arange(nodes + 1) * graph["h"]
        old_times = np.arange(warm_start["nodes"] + 1) * warm_start["h"]
        graph["opti"].set_initial(graph["X"], np.array([np.interp(new_times, old_times, row) for row in warm_start["x"]]))
        graph["opti"].set_initial(graph["U"], np.array([np.interp(new_times[:-1] + graph["h"] / 2, old_times[:-1] + warm_start["h"] / 2, row) for row in warm_start["u"]]))
    solution, report = checked_solve(graph["opti"], "Reference OCP")
    jacobian = ca.jacobian(graph["opti"].g, graph["opti"].x)
    return {
        "x": solution.value(graph["X"]), "u": solution.value(graph["U"]),
        "slack": np.asarray(solution.value(graph["S"])).ravel(),
        "h": graph["h"], "nodes": nodes, "objective": solution.value(graph["opti"].f),
        "stats": report, "jacobian_shape": jacobian.shape, "jacobian_nnz": jacobian.nnz(),
    }

### Validate continuous interpolation independently of the NLP slack

On each interval, reconstruct the exact surrogate motion
$q(\tau)=q_k+\tau v_k+\tfrac12\tau^2u_k$, $v(\tau)=v_k+\tau u_k$.
Evaluate **true simulated geometry**, not the fitted geometry used by the solver,
at twelve substeps per interval. Report signed clearance
$\|p_{\rm true}(q)-c\|-r_{\rm obstacle}$ and the dynamics defect separately.
The deliberately straight reference should collide; the optimized reference
should have positive sampled clearance. This is a numerical validation grid,
not a continuous-time or full-link safety certificate.

In [ ]:
def dense_trajectory(result, theta, samples_per_interval=12):
    dense_states, dense_time = [], []
    for index in range(result["nodes"]):
        for tau in np.linspace(0, result["h"], samples_per_interval, endpoint=False):
            state = result["x"][:, index]
            acceleration = result["u"][:, index]
            dense_states.append(np.r_[state[:6] + tau * state[6:] + 0.5 * tau**2 * acceleration, state[6:] + tau * acceleration])
            dense_time.append(index * result["h"] + tau)
    dense_states.append(result["x"][:, -1])
    dense_time.append(result["nodes"] * result["h"])
    dense_states = np.asarray(dense_states).T
    tools = np.array([np.asarray(position_map(q, theta)).ravel() for q in dense_states[:6].T]).T
    clearance = np.linalg.norm(tools - obstacle_center[:, None], axis=0) - obstacle_radius
    return np.asarray(dense_time), dense_states, tools, clearance


def reference_report(result):
    _, _, _, clearance = dense_trajectory(result, theta_truth)
    a_matrix, b_matrix = discrete_matrices(result["h"])
    defects = result["x"][:, 1:] - a_matrix @ result["x"][:, :-1] - b_matrix @ result["u"]
    print(f"N={result['nodes']}: J={result['objective']:.5f}, max normalized slack={np.max(result['slack']):.3g}, true dense clearance={np.min(clearance):.4f} m, dynamics defect={np.max(np.abs(defects)):.2g}")
    print("Constraint Jacobian shape / nnz:", result["jacobian_shape"], result["jacobian_nnz"])
    assert result["stats"]["success"]
    assert np.isfinite(result["x"]).all() and np.isfinite(result["u"]).all()
    assert np.max(np.abs(defects)) < 1e-6
    assert np.min(clearance) > 0
    assert np.max(np.abs(result["u"]) - u_max[:, None]) < 1e-6
    assert np.max(np.abs(result["x"][6:]) - v_max[:, None]) < 1e-6
    assert max_bound_violation(result["x"][:6], q_min_encoder[:, None].repeat(result["nodes"] + 1, axis=1), q_max_encoder[:, None].repeat(result["nodes"] + 1, axis=1)) < 1e-6
    np.testing.assert_allclose(result["x"][:, 0], np.r_[q_initial, np.zeros(6)], atol=1e-6)
    np.testing.assert_allclose(result["x"][:, -1], np.r_[q_goal, np.zeros(6)], atol=1e-6)
    return float(np.min(clearance))


reference = solve_reference_ocp(15)
reference_clearance = reference_report(reference)
nominal_path = np.array([np.asarray(position_map(q_initial + fraction * (q_goal - q_initial), theta_estimate)).ravel() for fraction in np.linspace(0, 1, 201)]).T
print("Straight joint-path clearance [m] (should collide):", np.min(np.linalg.norm(nominal_path - obstacle_center[:, None], axis=0) - obstacle_radius))

### Refine the mesh and compare the same continuous-time objective

Solve $N=24$ using interpolated positions, velocities, and interval accelerations
from the $N=15$ solution. Interpolation is a warm start; it need not satisfy the
new defects exactly. Compare objective, true dense clearance, Jacobian dimensions,
nonzeros, and density. Time-weighted costs preserve their meaning when $N$ changes.
Different local obstacle-avoidance paths are possible, so do not assert monotonic
objective improvement from these nonconvex solves.

In [ ]:
fine_reference = solve_reference_ocp(24, warm_start=reference)
fine_clearance = reference_report(fine_reference)
for result in (reference, fine_reference):
    rows, columns = result["jacobian_shape"]
    density = result["jacobian_nnz"] / (rows * columns)
    print(f"mesh {result['nodes']:2d}: Jacobian density {density:.4%}; objective {result['objective']:.5f}")
assert np.isfinite(fine_reference["objective"])

### Debug an actual infeasible transfer, then repair its cause

For a rest-to-rest scalar double integrator with acceleration bound $u_{\max}$,
$T\ge2\sqrt{|\Delta q|/u_{\max}}$. The largest joint bound is about 1 s here;
0.4 s is therefore impossible even before considering the obstacle or velocity
limits. Build and solve that impossible Opti problem below, catch the expected
failure, and inspect the worst constraint rows and source locations.

Repair the **duration** to 3 s while retaining the target, geometry, and limits,
then rerun on the same eight-interval mesh and validate actual dense clearance.
More IPOPT iterations cannot make the original transfer reachable. An iteration
limit alone would not prove infeasibility; the independent reachability bound
provides the explanation. The caught failure is deliberate and does not interrupt
the rest of the notebook.

In [ ]:
minimum_acceleration_duration = np.max(2 * np.sqrt(np.abs(q_goal - q_initial) / u_max))
impossible_duration = 0.4
print("Acceleration-only rest-to-rest lower bound [s]:", minimum_acceleration_duration)
assert impossible_duration < minimum_acceleration_duration
failed_reference = build_reference_ocp(nodes=8, total_time=impossible_duration)
failed_reference["opti"].solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 120})
try:
    checked_solve(failed_reference["opti"], "Expected too-short transfer failure")
except RuntimeError as error:
    expected_failure_status = solver_summary(failed_reference["opti"].stats())
    print("Caught expected infeasible-transfer failure:", expected_failure_status["return_status"])
else:
    raise AssertionError("A transfer below the acceleration reachability bound was accepted")

# Repair the duration, preserve the same geometry, target and limits, and solve again.
repaired_reference = solve_reference_ocp(8)
reference_report(repaired_reference)
print("Repair accepted at duration [s]:", duration)

### Formulate feedback MPC around the reference, using estimated states

Use $H=12$ prediction intervals and $h=0.2$ s. The MPC initial state is
$x_{0|k}=\hat x_k$ from MHE, never the simulated truth. The optimizer repeats
the exact dynamics, limits, calibrated tool obstacle constraints, and dimensionless
slacks. Its objective tracks the reference posture and velocity:

$$h\sum_iw_i\left(\|(q_i-q_i^{\rm ref})/0.3\|^2+
0.2\|(v_i-v_i^{\rm ref})/0.6\|^2+10^5s_i^2\right)
+0.02h\sum_i\|u_i/2\|^2+V_f.$$

$V_f$ adds terminal position/velocity penalties (25 and 5 using the same scales).
There is no hard terminal equality: newly observed disturbances should not turn
every feedback solve into a strict fixed-time endpoint problem. After the 3 s
reference ends, repeat its final rest state. These choices demonstrate feedback;
they do not establish a formal recursive-feasibility or stability theorem.

In [ ]:
def build_mpc(horizon=12):
    problem = ca.Opti()
    states = problem.variable(12, horizon + 1)
    inputs = problem.variable(6, horizon)
    slack = problem.variable(1, horizon + 1)
    initial_state = problem.parameter(12)
    target_states = problem.parameter(12, horizon + 1)
    problem.subject_to(states[:, 0] == initial_state)
    problem.subject_to(problem.bounded(q_min_encoder[:, None], states[:6, :], q_max_encoder[:, None]))
    problem.subject_to(problem.bounded(-v_max[:, None], states[6:, :], v_max[:, None]))
    problem.subject_to(problem.bounded(-u_max[:, None], inputs, u_max[:, None]))
    problem.subject_to(slack >= 0)
    for index in range(horizon):
        problem.subject_to(states[:, index + 1] == exact_step(states[:, index], inputs[:, index]))
        midpoint = exact_step(states[:, index], inputs[:, index], sample_time / 2)
        problem.subject_to(ca.sumsqr(position_map(midpoint[:6], theta_estimate) - obstacle_center) / planning_radius**2 + slack[0, index] >= 1)
    for index in range(horizon + 1):
        problem.subject_to(ca.sumsqr(position_map(states[:6, index], theta_estimate) - obstacle_center) / planning_radius**2 + slack[0, index] >= 1)
    node_weights = ca.DM([0.5, *([1.0] * (horizon - 1)), 0.5]).T
    state_stage = ca.sum1(((states[:6, :] - target_states[:6, :]) / 0.3)**2)
    state_stage += 0.2 * ca.sum1(((states[6:, :] - target_states[6:, :]) / 0.6)**2)
    objective = sample_time * ca.sum2(state_stage * node_weights)
    objective += 0.02 * sample_time * ca.sumsqr(inputs / 2)
    objective += 1e5 * sample_time * ca.sum2(slack**2 * node_weights)
    objective += 25 * ca.sumsqr((states[:6, -1] - target_states[:6, -1]) / 0.3)
    objective += 5 * ca.sumsqr((states[6:, -1] - target_states[6:, -1]) / 0.6)
    labels = ["estimated initial state"] * 12
    labels += ["joint angle bound"] * (6 * (horizon + 1)) + ["joint velocity bound"] * (6 * (horizon + 1))
    labels += ["acceleration bound"] * (6 * horizon) + ["nonnegative obstacle slack"] * (horizon + 1)
    for index in range(horizon):
        labels += [f"dynamics interval {index}"] * 12 + [f"midpoint obstacle interval {index}"]
    labels += [f"tool obstacle node {index}" for index in range(horizon + 1)]
    problem.tutorial_constraint_labels = labels
    problem.minimize(objective)
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 150, "tol": 1e-7})
    return {"opti": problem, "X": states, "U": inputs, "S": slack, "initial": initial_state, "reference": target_states, "horizon": horizon}


def reference_window(result, first, horizon):
    indices = np.minimum(np.arange(first, first + horizon + 1), result["nodes"])
    return result["x"][:, indices]

### Close the loop: truth → noisy sensors → MHE → MPC → first input

Simulate 24 feedback moves. A stationary six-interval history initializes MHE;
each subsequent move uses the latest encoders, tool position, and known commands.
MHE's last state initializes MPC. Apply **only the first** predicted acceleration
to a separate disturbed plant, then measure again. Both graph objects are reused;
parameters and warm starts change between solves.

Truth is used by the plant, plots, and validation only. Dense within-step clearance
uses true calibration and the actual disturbed acceleration. Report this separately
from predicted slacks and constraint defects. The default should reach within
3 cm of the Cartesian target while respecting commanded acceleration limits and
keeping positive sampled tool clearance. A solver failure prints Opti infeasibility
details and raises; it must not be silently counted as successful control.

In [ ]:
def run_closed_loop(noise_multiplier=1.0, moves=24):
    rng = np.random.default_rng(44)
    window = 6
    estimator = build_mhe(theta_estimate, window, 0.003 * noise_multiplier, 0.001 * noise_multiplier)
    controller = build_mpc(12)
    state = np.r_[q_initial, np.zeros(6)]

    def observe(current):
        encoders = current[:6] + rng.normal(0, 0.003 * noise_multiplier, 6)
        tool = np.asarray(position_map(current[:6], theta_truth)).ravel() + rng.normal(0, 0.001 * noise_multiplier, 3)
        return encoders, tool

    # Six stationary samples create a real observation history before motion starts.
    q_history, p_history = [], []
    for _ in range(window + 1):
        encoders, tool = observe(state)
        q_history.append(encoders)
        p_history.append(tool)
    u_history = [np.zeros(6) for _ in range(window)]
    arrival = np.r_[q_history[0], np.zeros(6)]
    previous_estimates = None
    previous_prediction = None
    truth, estimates, commands, slacks, statuses = [state.copy()], [], [], [], []
    true_clearances, clearance_times, prediction_defects = [], [], []
    mhe_reports, mpc_reports = [], []
    for move in range(moves):
        estimate, fitted_states, estimator_stats = estimate_window(
            estimator, np.asarray(u_history[-window:]).T,
            np.asarray(q_history[-window - 1:]).T, np.asarray(p_history[-window - 1:]).T,
            arrival, previous_estimates,
        )
        if not estimator_stats["accepted"]:
            raise RuntimeError("Rejected state estimate before feeding MPC")
        arrival, previous_estimates = fitted_states[:, 1], fitted_states
        target = reference_window(reference, move, controller["horizon"])
        problem = controller["opti"]
        problem.set_value(controller["initial"], estimate)
        problem.set_value(controller["reference"], target)
        if previous_prediction is None:
            problem.set_initial(controller["X"], target)
            problem.set_initial(controller["U"], 0)
            problem.set_initial(controller["S"], 0)
        else:
            predicted_x, predicted_u, predicted_s = previous_prediction
            problem.set_initial(controller["X"], np.column_stack([predicted_x[:, 1:], a_discrete @ predicted_x[:, -1]]))
            problem.set_initial(controller["U"], np.column_stack([predicted_u[:, 1:], np.zeros(6)]))
            problem.set_initial(controller["S"], np.r_[predicted_s[1:], predicted_s[-1]][None, :])
        solution, controller_report = checked_solve(problem, f"MPC move {move}")
        predicted_x = solution.value(controller["X"])
        predicted_u = solution.value(controller["U"])
        predicted_s = np.asarray(solution.value(controller["S"])).ravel()
        prediction_defect = float(np.max(np.abs(predicted_x[:, 1:] - a_discrete @ predicted_x[:, :-1] - b_discrete @ predicted_u)))
        input_violation = float(np.max(np.maximum(np.abs(predicted_u) - u_max[:, None], 0)))
        controller_report.update({"dynamics_defect": prediction_defect, "input_violation": input_violation, "maximum_slack": float(np.max(predicted_s))})
        # These checks precede the FIRST use of a predicted command by the plant.
        if prediction_defect > 1e-5 or input_violation > 1e-5:
            print("Rejected MPC candidate:", controller_report)
            raise RuntimeError("MPC candidate failed independent dynamics/input acceptance")
        previous_prediction = predicted_x, predicted_u, predicted_s
        command = predicted_u[:, 0]
        disturbance = rng.normal(0, 0.04, 6)
        for tau in np.linspace(0, sample_time, 16):
            true_q = state[:6] + tau * state[6:] + 0.5 * tau**2 * (command + disturbance)
            tool = np.asarray(position_map(true_q, theta_truth)).ravel()
            true_clearances.append(np.linalg.norm(tool - obstacle_center) - obstacle_radius)
            clearance_times.append(move * sample_time + tau)
        state = a_discrete @ state + b_discrete @ (command + disturbance)
        q_sample, p_sample = observe(state)
        q_history.append(q_sample)
        p_history.append(p_sample)
        u_history.append(command)
        truth.append(state.copy())
        estimates.append(estimate)
        commands.append(command)
        slacks.append(np.max(predicted_s))
        statuses.append(bool(controller_report["accepted"] and estimator_stats["accepted"]))
        prediction_defects.append(prediction_defect)
        mhe_reports.append(estimator_stats)
        mpc_reports.append(controller_report)
    return {
        "x": np.asarray(truth).T, "estimate": np.asarray(estimates).T, "u": np.asarray(commands).T,
        "slack": np.asarray(slacks), "clearance": np.asarray(true_clearances), "clearance_time": np.asarray(clearance_times),
        "success": all(statuses), "max_defect": max(prediction_defects), "h": sample_time,
        "nodes": moves, "q_measured": np.asarray(q_history[window:]).T,
        "mhe_reports": mhe_reports, "mpc_reports": mpc_reports,
    }


closed_loop = run_closed_loop()
final_tool = np.asarray(position_map(closed_loop["x"][:6, -1], theta_truth)).ravel()
desired_tool = np.asarray(position_map(q_goal, theta_estimate)).ravel()
final_tool_error = np.linalg.norm(final_tool - desired_tool)
print("All MHE/MPC solves successful:", closed_loop["success"])
print("Final true Cartesian target error [m]:", final_tool_error)
print("Minimum true sampled tool clearance [m]:", np.min(closed_loop["clearance"]))
print("Maximum normalized obstacle slack:", np.max(closed_loop["slack"]))
assert closed_loop["success"] and np.isfinite(closed_loop["x"]).all()
assert closed_loop["max_defect"] < 1e-5
assert np.max(np.abs(closed_loop["u"]) - u_max[:, None]) < 1e-5
assert np.min(closed_loop["clearance"]) > 0
assert final_tool_error < 0.03

### Inspect the motion, controls, and actual obstacle clearance

Rotate and zoom the 3D Plotly view, drag its time slider, or press **Play** to
inspect the arm and tool path. These controls replay computed results and do not
resolve an optimization. Time plots show all encoder positions, joint velocities,
and interval-held acceleration commands; hover for values and click legend entries
to isolate joints. The clearance plot uses metres, whereas optimizer slack is
dimensionless. The static fallback shows five arm snapshots and the same trajectories.

Set `CASADI_TUTORIAL_PLOTS=static` for deterministic Matplotlib output. Plotting
receives frame origins calculated above and never imports a hidden robot model.

In [ ]:
def render_trajectory(result, theta=theta_estimate, title="Robot motion"):
    times = np.arange(result["nodes"] + 1) * result["h"]
    tools = np.array([np.asarray(position_map(q, theta)).ravel() for q in result["x"][:6].T]).T
    use_plotly = os.environ.get("CASADI_TUTORIAL_PLOTS", "auto").lower() != "static"
    try:
        if not use_plotly:
            raise ImportError
        import plotly.graph_objects as go
    except ImportError:
        axis = plt.figure(figsize=(8, 6)).add_subplot(projection="3d")
        axis.plot(*tools, label="tool trace")
        for index in np.unique(np.linspace(0, result["nodes"], 5).astype(int)):
            points = np.asarray(origins_map(result["x"][:6, index], theta))
            axis.plot(*points, "o-", alpha=0.45)
        plot_robot(np.asarray(origins_map(result["x"][:6, -1], theta)), ax=axis, obstacle=(obstacle_center, obstacle_radius), title=title, backend="matplotlib")
        display_plot(axis)
    else:
        initial = np.asarray(origins_map(result["x"][:6, 0], theta))
        azimuth, polar = np.mgrid[0:2 * np.pi:24j, 0:np.pi:12j]
        sphere = [obstacle_center[j] + obstacle_radius * component for j, component in enumerate((np.cos(azimuth) * np.sin(polar), np.sin(azimuth) * np.sin(polar), np.cos(polar)))]
        figure = go.Figure(data=[
            go.Scatter3d(x=initial[0], y=initial[1], z=initial[2], mode="lines+markers", name="arm"),
            go.Scatter3d(x=tools[0], y=tools[1], z=tools[2], mode="lines", name="tool path"),
            go.Surface(x=sphere[0], y=sphere[1], z=sphere[2], opacity=0.35, showscale=False, name="obstacle"),
        ])
        figure.frames = [go.Frame(name=str(index), data=[go.Scatter3d(x=points[0], y=points[1], z=points[2])], traces=[0]) for index, points in enumerate(np.asarray([origins_map(q, theta) for q in result["x"][:6].T]))]
        figure.update_layout(title=title, scene=dict(aspectmode="data", xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]"),
            updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate", args=[None, {"frame": {"duration": 100}, "fromcurrent": True}]), dict(label="Pause", method="animate", args=[[None], {"mode": "immediate"}])])],
            sliders=[dict(steps=[dict(label=f"{time:.1f}s", method="animate", args=[[str(index)], {"mode": "immediate", "frame": {"duration": 0}}]) for index, time in enumerate(times)])])
        display_plot(figure)
    labels = [f"joint {joint + 1}" for joint in range(6)]
    display_plot(plot_series(times, result["x"][:6], title=title + ": encoder positions", ylabel="q [rad]", labels=labels))
    display_plot(plot_series(times, result["x"][6:], title=title + ": velocities", ylabel="v [rad/s]", labels=labels))
    # Append the final input for a correctly held last interval in step plots.
    held_inputs = np.column_stack([result["u"], result["u"][:, -1]])
    display_plot(plot_series(times, held_inputs, title=title + ": acceleration commands", ylabel="u [rad/s²]", labels=labels, steps=True))


render_trajectory(reference, title="Optimized reference and active obstacle")
render_trajectory(closed_loop, title="Estimated-state feedback motion")
dense_time, _, _, dense_clearance = dense_trajectory(reference, theta_truth)
display_plot(plot_series(dense_time, {"true tool clearance": dense_clearance, "collision boundary": np.zeros_like(dense_clearance)}, title="Reference clearance independent of slack", ylabel="signed clearance [m]"))

### Monitor accepted solves and distinguish geometry error from control error

Retain every MHE/MPC status, iteration count, measured wall time, constraint
violation, independent dynamics defect, and maximum predicted slack. The gates
above checked these candidates **before** an estimate entered MPC or an acceleration
entered the plant. These plots reveal isolated difficult solves that aggregate
success flags would hide. Timing depends on the machine and environment.

The first plot checks true within-step feedback clearance, not just the planned
reference. The velocity plot shows the estimate actually used by MPC. Finally,
evaluate nominal and calibrated FK along the **same executed encoder trajectory**
against true simulated tool positions. This isolates geometric prediction accuracy;
it is not a second controller run or a claim of improved closed-loop performance
under all disturbances. Final Cartesian target error was scored separately above.

In [ ]:
feedback_time = sample_time * np.arange(closed_loop["nodes"])
display_plot(plot_series(closed_loop["clearance_time"], {
    "true closed-loop tool clearance": closed_loop["clearance"],
    "collision boundary": np.zeros_like(closed_loop["clearance"]),
}, title="Feedback: dense true clearance, independent of predicted slack", ylabel="clearance [m]"))
for field, title, units in (
    ("iterations", "Solver iteration counts", "iterations"),
    ("wall_time_s", "Measured solve wall times", "elapsed [s]"),
    ("constraint_violation", "Original-constraint violations", "maximum violation"),
    ("dynamics_defect", "Independent discrete-dynamics defects", "maximum defect"),
):
    display_plot(plot_series(feedback_time, {
        "MHE": np.array([report[field] for report in closed_loop["mhe_reports"]]),
        "MPC": np.array([report[field] for report in closed_loop["mpc_reports"]]),
    }, title=title, ylabel=units))
display_plot(plot_series(feedback_time, {"MPC maximum predicted slack": closed_loop["slack"]}, title="Normalized obstacle relaxation at each feedback solve", ylabel="dimensionless slack"))
print("MHE statuses:", sorted({report["return_status"] for report in closed_loop["mhe_reports"]}))
print("MPC statuses:", sorted({report["return_status"] for report in closed_loop["mpc_reports"]}))
display_plot(plot_series(feedback_time, {
    "true joint-2 velocity": closed_loop["x"][7, :-1],
    "MHE joint-2 velocity": closed_loop["estimate"][7],
}, title="The state estimate actually used by the controller", ylabel="v2 [rad/s]"))

path_time = sample_time * np.arange(closed_loop["nodes"] + 1)
actual_tools = np.array([np.asarray(position_map(q, theta_truth)).ravel() for q in closed_loop["x"][:6].T]).T
nominal_tools = np.array([np.asarray(position_map(q, nominal_theta)).ravel() for q in closed_loop["x"][:6].T]).T
calibrated_tools = np.array([np.asarray(position_map(q, theta_estimate)).ravel() for q in closed_loop["x"][:6].T]).T
nominal_geometry_error = np.linalg.norm(nominal_tools - actual_tools, axis=0)
calibrated_geometry_error = np.linalg.norm(calibrated_tools - actual_tools, axis=0)
display_plot(plot_series(path_time, {
    "nominal FK versus true tool": 1000 * nominal_geometry_error,
    "calibrated FK versus true tool": 1000 * calibrated_geometry_error,
}, title="Same executed path: geometric prediction error", ylabel="tool-position error [mm]"))
print("Same-path position RMSE: nominal / calibrated [m]:", np.sqrt(np.mean(nominal_geometry_error**2)), "/", np.sqrt(np.mean(calibrated_geometry_error**2)))
assert np.sqrt(np.mean(calibrated_geometry_error**2)) < np.sqrt(np.mean(nominal_geometry_error**2))

### Complete the mesh comparison with fine-grid plots and a reachability diagnosis

The same plotting code shows every state and held input on the fine mesh.
For a rest-to-rest scalar double integrator with acceleration bound $u_{\max}$,
the fastest unconstrained transfer takes $2\sqrt{|\Delta q|/u_{\max}}$.
The largest per-joint bound below rules out a proposed 0.4 s transfer before
invoking IPOPT. Obstacle, velocity, or posture constraints can only increase
the required time. Increasing `max_iter` cannot fix this infeasibility;
extending the duration to 3 s is the concrete repair used above.

In [ ]:
render_trajectory(fine_reference, title="Fine-mesh worked solution")
fine_time, _, _, fine_clearance_trace = dense_trajectory(fine_reference, theta_truth)
display_plot(plot_series(fine_time, {"true clearance": fine_clearance_trace, "collision boundary": np.zeros_like(fine_clearance_trace)}, title="Fine mesh: dense clearance", ylabel="clearance [m]"))
rest_to_rest_lower_bound = np.max(2 * np.sqrt(np.abs(q_goal - q_initial) / u_max))
proposed_duration = 0.4
print("Acceleration-only duration lower bound [s]:", rest_to_rest_lower_bound)
print("Proposed infeasible duration [s]:", proposed_duration)
assert proposed_duration < rest_to_rest_lower_bound < duration

### Reading the results

A complete solution checks model derivatives, held-out identification error,
estimation error, exact dynamics, endpoint/input limits, sparsity, and true
sampled clearance. The optimizer's obstacle slack is dimensionless and cannot
substitute for geometric validation. Calibration offsets enter FK once;
encoder states remain in the same coordinate convention throughout MHE/MPC.
The last-joint position derivative is zero because this DH tool origin lies
on its rotation axis; the angular derivative is not zero.